# Senior Project Regression Report Demo

## Setup Data

In [9]:
from pathlib import Path

from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from mlreport import ComparisonReport, Report

Path("reports").mkdir(exist_ok=True)

X, y = load_diabetes(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
author = "Lucas Summers"
theme = "light"

## Model 1: Train/Test Split Report

In [4]:
split_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]
)

split_model.fit(X_train, y_train)
y_pred_train = split_model.predict(X_train)
y_pred_test = split_model.predict(X_test)

split_report = Report(
    split_model,
    title="Diabetes Regression Report - Train/Test Split",
    author=author,
    description="Linear Regression evaluated on a standard train/test split.",
    theme=theme,
)

split_report.add_split("train", X_train, y_train, y_pred_train)
split_report.add_split("test", X_test, y_test, y_pred_test)
split_report.build().to_pdf("reports/diabetes_split_report.pdf").summary()

Report
  title: Diabetes Regression Report - Train/Test Split
  author: Lucas Summers
  description: Linear Regression evaluated on a standard train/test split.
  generated_at: 2026-04-19 19:23

Model
  name: LinearRegression
  type: Regression
  sklearn_version: 1.8.0
  param_count: 13

Data
  features: 10
  splits: train=331, test=111
  total: 442

Metrics
  Mean Absolute Error: train=44.0548, test=41.5485
  Max Error: train=163.9605, test=153.6572
  Median Absolute Error: train=38.9980, test=35.2108
  Mean Squared Error: train=2907.2578, test=2848.3107
  R² Score: train=0.5190, test=0.4849
  Root Mean Squared Error: train=53.9190, test=53.3696


## Model 2: CV Report

In [5]:
cv_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    random_state=42,
)
cv_model.fit(X_train, y_train)

cv_report = Report(
    cv_model,
    title="Diabetes Regression Report - Cross-Validation",
    author=author,
    description="Random Forest evaluated with 5-fold cross-validation.",
    theme=theme,
)

cv_report.add_crossval(X_train, y_train, cv=cv)
cv_report.build().to_pdf("reports/diabetes_cv_report.pdf").summary()

Report
  title: Diabetes Regression Report - Cross-Validation
  author: Lucas Summers
  description: Random Forest evaluated with 5-fold cross-validation.
  generated_at: 2026-04-19 19:23

Model
  name: RandomForestRegressor
  type: Regression
  sklearn_version: 1.8.0
  param_count: 18

Data
  features: 10
  splits: cv=331
  total: 331
  cv_folds: 5

Metrics
  Mean Absolute Error: cv=49.7012 (std=3.0789)
  Max Error: cv=142.7364 (std=7.9678)
  Median Absolute Error: cv=44.9407 (std=5.2820)
  Mean Squared Error: cv=3637.4399 (std=414.3230)
  R² Score: cv=0.3763 (std=0.1192)
  Root Mean Squared Error: cv=60.2124 (std=3.4502)


## Model 3: Tuning + CV Report

In [6]:
search = GridSearchCV(
    estimator=Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", KNeighborsRegressor()),
        ]
    ),
    param_grid={
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"],
    },
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

search.fit(X_train, y_train)
best_tuned_model = search.best_estimator_

tuned_cv_report = Report(
    best_tuned_model,
    title="Diabetes Regression Report - Tuning + Cross-Validation",
    author=author,
    description="KNN regressor tuned with GridSearchCV and evaluated with 5-fold CV.",
    theme=theme,
)

tuned_cv_report.add_crossval(X_train, y_train, cv=cv)
tuned_cv_report.add_search(search)
tuned_cv_report.build().to_pdf("reports/diabetes_tuned_cv_report.pdf").summary()

Report
  title: Diabetes Regression Report - Tuning + Cross-Validation
  author: Lucas Summers
  description: KNN regressor tuned with GridSearchCV and evaluated with 5-fold CV.
  generated_at: 2026-04-19 19:23

Model
  name: KNeighborsRegressor
  type: Regression
  sklearn_version: 1.8.0
  param_count: 16
  tuning_method: GridSearchCV
  tuning_metric: neg_root_mean_squared_error
  tuning_best_score: -61.4718
  tuning_candidates: 8
  tuning_cv_folds: 5
  tuning_best_params: model__n_neighbors=9, model__weights=distance

Data
  features: 10
  splits: cv=331
  total: 331
  cv_folds: 5

Metrics
  Mean Absolute Error: cv=49.3585 (std=3.0074)
  Max Error: cv=155.8950 (std=7.9642)
  Median Absolute Error: cv=42.9693 (std=5.3624)
  Mean Squared Error: cv=3790.9658 (std=417.7825)
  R² Score: cv=0.3535 (std=0.0975)
  Root Mean Squared Error: cv=61.4718 (std=3.4901)


## Comparison Report of 3 Models

In [7]:
comparison = ComparisonReport(
    reports=[
        split_report,
        cv_report,
        tuned_cv_report,
    ],
    title="Diabetes Workflow Comparison",
    author=author,
    description="Comparison of three regression workflows on the Diabetes dataset.",
    theme=theme,
)

comparison.build().to_pdf("reports/diabetes_comparison_report.pdf").summary()

Comparison Report
  title: Diabetes Workflow Comparison
  author: Lucas Summers
  description: Comparison of three regression workflows on the Diabetes dataset.
  baseline: LinearRegression

Models
  1. LinearRegression (LinearRegression) [baseline]: Linear Regression evaluated on a standard train/test split.
  2. RandomForestRegressor (RandomForestRegressor): Random Forest evaluated with 5-fold cross-validation.
  3. KNeighborsRegressor (KNeighborsRegressor): KNN regressor tuned with GridSearchCV and evaluated with 5-fold CV.

Metrics
  Mean Absolute Error: LinearRegression=41.5485, RandomForestRegressor=49.7012 (+8.1527), KNeighborsRegressor=49.3585 (+7.8100), best=RandomForestRegressor
  Max Error: LinearRegression=153.6572, RandomForestRegressor=142.7364 (-10.9208), KNeighborsRegressor=155.8950 (+2.2378), best=KNeighborsRegressor
  Median Absolute Error: LinearRegression=35.2108, RandomForestRegressor=44.9407 (+9.7299), KNeighborsRegressor=42.9693 (+7.7585), best=RandomForestRegres